# Task 4 — Real-World Data Project (Health Domain)
## Breast Cancer Diagnosis: End-to-End Data Analysis & Prediction

**Dataset:** Breast Cancer Wisconsin (Diagnostic) Data Set — a real clinical dataset (UCI ML Repository / scikit-learn) of measurements taken from digitized images of fine needle aspirate (FNA) biopsies of breast masses. Each row is one patient; the target is the diagnosis (Malignant / Benign).

**Goal:** Perform full exploratory data analysis, then train and evaluate machine learning models to predict whether a tumor is malignant or benign from its cell measurements.

**Pipeline:**
1. Load & save the dataset
2. Exploratory Data Analysis (EDA) with visualizations
3. Preprocessing (train/test split, scaling)
4. Model training (Logistic Regression, Random Forest)
5. Model evaluation (accuracy, confusion matrix, ROC-AUC)
6. Feature importance
7. Conclusions

> Run the cells from top to bottom (Run All in VS Code). Charts/results will also be saved into the `outputs/` folder, matching the pre-generated copies already included there.


## 1. Setup & Imports

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

sns.set_theme(style="whitegrid", palette="viridis")
%matplotlib inline

# Paths (this notebook lives in notebooks/, so go one level up to the project root)
BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, "data")
OUT_DIR = os.path.join(BASE_DIR, "outputs")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
DATA_CSV = os.path.join(DATA_DIR, "breast_cancer_data.csv")


## 2. Load the Dataset
We load the real breast cancer dataset and persist it as a CSV file, so the project works with a genuine on-disk dataset file (`data/breast_cancer_data.csv`), just like any other real-world data project.

In [ ]:
bunch = load_breast_cancer(as_frame=True)
df = bunch.frame.copy()
df["diagnosis"] = df["target"].map({0: "Malignant", 1: "Benign"})
df.to_csv(DATA_CSV, index=False)

print(f"Shape: {df.shape[0]} patients x {df.shape[1]} columns")
df.head()


## 3. Exploratory Data Analysis (EDA)

In [ ]:
print("Missing values per column (top 5):")
print(df.isnull().sum().sort_values(ascending=False).head())

print("\nClass balance:")
print(df["diagnosis"].value_counts())

df.describe()


### 3.1 Target Class Distribution

In [ ]:
plt.figure(figsize=(5, 4))
ax = sns.countplot(data=df, x="diagnosis", hue="diagnosis", legend=False)
ax.set_title("Diagnosis Class Distribution")
ax.set_xlabel("")
ax.set_ylabel("Number of Patients")
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "01_class_distribution.png"), dpi=150)
plt.show()


### 3.2 Correlation Heatmap
Looking at how the 'mean' tumor measurements relate to one another.

In [ ]:
mean_cols = [c for c in df.columns if c.startswith("mean ")]
plt.figure(figsize=(10, 8))
corr = df[mean_cols].corr()
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, square=True)
plt.title("Correlation Heatmap - 'Mean' Tumor Measurements")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "02_correlation_heatmap.png"), dpi=150)
plt.show()


### 3.3 Feature Distributions by Diagnosis

In [ ]:
key_features = ["mean radius", "mean texture", "mean perimeter",
                 "mean area", "mean smoothness", "mean concavity"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.flatten(), key_features):
    sns.histplot(data=df, x=feat, hue="diagnosis", kde=True, ax=ax, alpha=0.5)
    ax.set_title(feat)
plt.suptitle("Key Feature Distributions by Diagnosis", y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "03_feature_distributions.png"), dpi=150)
plt.show()


### 3.4 Boxplots (Outlier Inspection)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, feat in zip(axes, ["mean radius", "mean area", "mean concavity"]):
    sns.boxplot(data=df, x="diagnosis", y=feat, hue="diagnosis", legend=False, ax=ax)
    ax.set_title(f"{feat} by Diagnosis")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "04_boxplots.png"), dpi=150)
plt.show()


**EDA takeaway:** Malignant tumors tend to have visibly larger and more irregular measurements (radius, perimeter, area, concavity) than benign ones, and several of the 'mean' features are highly correlated with each other.

## 4. Preprocessing: Train/Test Split + Scaling

In [ ]:
X = df.drop(columns=["target", "diagnosis"])
y = df["target"]  # 0 = Malignant, 1 = Benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")


## 5. Model Training
We train two complementary models: a simple, interpretable **Logistic Regression**, and a more flexible **Random Forest**.

In [ ]:
log_reg = LogisticRegression(max_iter=5000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train_scaled, y_train)

models = {"Logistic Regression": log_reg, "Random Forest": rf}
print("Models trained:", list(models.keys()))


## 6. Model Evaluation

In [ ]:
results = {}
roc_fig = plt.figure(figsize=(6, 5))

for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results[name] = {"accuracy": round(acc, 4), "precision": round(prec, 4),
                      "recall": round(rec, 4), "f1_score": round(f1, 4),
                      "roc_auc": round(auc, 4)}

    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=["Malignant", "Benign"]))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4.5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Malignant", "Benign"], yticklabels=["Malignant", "Benign"])
    plt.title(f"Confusion Matrix - {name}")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.tight_layout()
    safe_name = name.lower().replace(" ", "_")
    plt.savefig(os.path.join(OUT_DIR, f"05_confusion_matrix_{safe_name}.png"), dpi=150)
    plt.show()

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.figure(roc_fig.number)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.figure(roc_fig.number)
plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "06_roc_curve_comparison.png"), dpi=150)
plt.show()

pd.DataFrame(results).T


## 7. Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
top10 = importances.sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 6))
sns.barplot(x=top10.values, y=top10.index, hue=top10.index, legend=False)
plt.title("Top 10 Most Important Features (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "07_feature_importance.png"), dpi=150)
plt.show()

top10


## 8. Save Model Artifacts & Results

In [ ]:
joblib.dump(rf, os.path.join(OUT_DIR, "random_forest_model.joblib"))
joblib.dump(scaler, os.path.join(OUT_DIR, "feature_scaler.joblib"))

with open(os.path.join(OUT_DIR, "model_results.json"), "w") as f:
    json.dump(results, f, indent=2)

print("Saved trained models, scaler, and metrics to outputs/")


## 9. Conclusions

1. **Class balance:** the dataset is moderately imbalanced (~63% Benign / 37% Malignant); a stratified split preserved this ratio between train and test sets.
2. **Correlated features:** mean radius, perimeter, area, and concavity are strongly correlated with each other and with malignancy — larger, more irregular cells point to malignant tumors.
3. **Model performance:** both Logistic Regression and Random Forest achieve strong accuracy and ROC-AUC scores (see the metrics table above); the model with the higher ROC-AUC is the stronger overall classifier on this dataset.
4. **Most predictive features:** the "worst" and "mean" versions of perimeter, area, radius, and concave points consistently rank as the top predictors.
5. **Clinical priority:** recall on the Malignant class matters most in this domain — a missed cancer case (false negative) is far costlier than a false alarm, so the probability threshold should be tuned with that asymmetry in mind before any real deployment.

This notebook, the saved CSV dataset, and all charts/metrics in `outputs/` together form a complete, reproducible, end-to-end applied data science workflow on a real health dataset.
